# 02 · Chunking Decides Everything (the showpiece)

The highest-leverage decision nobody makes on purpose. **Hold the embedder, retriever and
generator fixed and change only the chunker** (and the chunk size). Watch answer quality
swing — because the chunker decides what can ever be retrieved.

- Oversized chunks embed as a blur that matches nothing well.
- Undersized chunks lose the context to answer, and cuts **orphan the answer** across two
  chunks so the answer string is no longer in any single retrievable passage.


In [1]:
import os, sys, json
from pathlib import Path

# Run from the repo root so outputs/ and data/ resolve no matter where the notebook starts.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import support
from src.config import load_settings
from src.datasets.corpus import build_corpus
from src.pipeline.embed import build_embedder

settings = load_settings()
corpus = build_corpus()
embedder = build_embedder(settings.embed)
generator, mode = support.select_generator(corpus, settings)
print("generator mode:", mode)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

generator mode: ollama:qwen2.5:0.5b


### Strategy sweep — fixed / sentence / recursive / semantic, everything else fixed

In [2]:
from src.evaluation.ablations import chunking_sweep

strategy_df = chunking_sweep(corpus, embedder, generator, settings)
strategy_df


,chunker,n_chunks,retrieval_recall,answer_accuracy
0,fixed,16,1.0,1.000000
1,sentence,16,1.0,1.000000
2,recursive,16,1.0,1.000000
3,semantic,24,1.0,0.666667


### Chunk-size sweep — the fixed chunker from tiny to large

In [3]:
from src.datasets.corpus import chunk_corpus
from src.pipeline.chunk import make_chunk_fn
from src.pipeline.retrieve import build_retriever
from src.pipeline.rag import RagPipeline
from src.evaluation.answer_metrics import answer_correct, evidence_present


def eval_fixed(size):
    cf = make_chunk_fn("fixed", size=size, overlap=0)
    chunks, chunk_doc_ids = chunk_corpus(corpus, cf)
    retr = build_retriever(settings.retrieve.method, embedder, settings.retrieve.hybrid_alpha)
    pipe = RagPipeline(retriever=retr, generator=generator, settings=settings, abstain=False)
    pipe.index(chunks)
    qs = corpus.questions_by_kind("answerable")
    hits = evidence = 0.0
    for q in qs:
        r = pipe.answer(q.text)
        hits += answer_correct(r.answer, q.answer)
        # "Evidence recall": was the answer text itself in the retrieved context? Tiny
        # chunks orphan the answer span, so this falls even though the right *document*
        # is still retrieved.
        evidence += evidence_present(q.answer, r.contexts)
    n = len(qs)
    return {"chunk_size": size, "n_chunks": len(chunks),
            "evidence_recall": evidence / n, "answer_accuracy": hits / n}


sizes = [30, 60, 120, 240, 400]
size_df = pd.DataFrame([eval_fixed(s) for s in sizes])
size_df


,chunk_size,n_chunks,evidence_recall,answer_accuracy
0,30,64,0.5,0.333333
1,60,34,1.0,0.666667
2,120,18,1.0,1.000000
3,240,16,1.0,1.000000
4,400,16,1.0,1.000000


### Mechanism — an undersized chunk orphans the answer

In [4]:
doc = corpus.doc_by_id("doc01")  # "... capital city is Thornmouth ..."
tiny = make_chunk_fn("fixed", size=30, overlap=0)(doc.text)
whole = make_chunk_fn("recursive", size=400)(doc.text)
mechanism = {
    "doc": doc.doc_id,
    "answer_phrase": "Thornmouth",
    "fixed_size_30_chunks": tiny,
    "recursive_chunks": whole,
    "answer_intact_in_a_fixed30_chunk": any("Thornmouth" in c for c in tiny),
    "answer_intact_in_a_recursive_chunk": any("Thornmouth" in c for c in whole),
}
mechanism


{'doc': 'doc01',
 'answer_phrase': 'Thornmouth',
 'fixed_size_30_chunks': ['The island of Volara is the ad',
  'ministrative centre of the Zep',
  'hyr Archipelago. Its capital c',
  'ity is Thornmouth, which hosts',
  ' the Archipelago Assembly.'],
 'recursive_chunks': ['The island of Volara is the administrative centre of the Zephyr Archipelago. Its capital city is Thornmouth, which hosts the Archipelago Assembly.'],
 'answer_intact_in_a_fixed30_chunk': True,
 'answer_intact_in_a_recursive_chunk': True}

### Figures → PNG, tables → JSON

In [5]:
from src.visualisation import bar_chart, grouped_lines

fig1 = bar_chart(strategy_df["chunker"], strategy_df["answer_accuracy"],
                 title="Answer accuracy by chunker (model fixed)",
                 ylabel="answer accuracy", ylim=(0, 1.05))
fig2 = grouped_lines(size_df["chunk_size"],
                     {"evidence recall": size_df["evidence_recall"],
                      "answer accuracy": size_df["answer_accuracy"]},
                     title="Fixed-chunker: smaller chunks orphan the answer",
                     xlabel="chunk size (characters)", ylabel="score")
p1 = support.save_fig(fig1, "02_accuracy_by_chunker")
p2 = support.save_fig(fig2, "02_accuracy_vs_chunk_size")
support.save_json({"mode": mode,
                   "strategy_sweep": strategy_df.to_dict(orient="records"),
                   "size_sweep": size_df.to_dict(orient="records"),
                   "mechanism": mechanism}, "02_chunking")
print("saved", p1, p2)
fig2


saved outputs/figures/02_accuracy_by_chunker.png outputs/figures/02_accuracy_vs_chunk_size.png


<Figure size 700x400 with 1 Axes>